In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# 1. Leer la tabla limpia Silver
silver_data = spark.table("silver_inventory_cleaned")

# 2. Agrupar facturación y volumen por SKU
sku_summary = silver_data.groupBy("StockCode", "Description").agg(
    F.sum("TotalValue").alias("SKU_TotalValue"),
    F.sum("Quantity").alias("SKU_TotalQuantity")
)

# 3. Obtener la facturación total global
total_revenue = sku_summary.select(F.sum("SKU_TotalValue")).collect()[0][0]

# 4. Calcular % acumulado y categorizar A, B o C mediante Window Functions
window_spec = Window.orderBy(F.col("SKU_TotalValue").desc())

gold_df = (
    sku_summary
    .withColumn("Pct_Revenue", F.col("SKU_TotalValue") / total_revenue)
    .withColumn("Cumulative_Pct", F.sum("Pct_Revenue").over(window_spec))
    .withColumn(
        "ABC_Class",
        F.when(F.col("Cumulative_Pct") <= 0.80, "A")
         .when(F.col("Cumulative_Pct") <= 0.95, "B")
         .otherwise("C")
    )
)

# 5. Guardar como Tabla Delta Gold
(
    gold_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("gold_inventory_abc")
)

# Resumen estadístico por clase ABC
abc_summary = (
    spark.table("gold_inventory_abc")
    .groupBy("ABC_Class")
    .agg(
        F.count("StockCode").alias("Total_SKUs"),
        F.round(F.sum("SKU_TotalValue"), 2).alias("Total_Revenue_USD"),
        F.round(F.avg("SKU_TotalValue"), 2).alias("Avg_Revenue_Per_SKU")
    )
    .orderBy("ABC_Class")
)

display(abc_summary)

# 6. Mostrar distribución final
display(spark.table("gold_inventory_abc").limit(15))